# [v1 Retrospective] Exploratory Data Analysis
> **What this notebook shows:** Univariate and bivariate exploration of network features, feature distributions across classes, and correlation heatmaps.
>
> **What went wrong in v1:**
> 1. Overlooked that id is an artificial row index strongly correlated with target classes due to ordered time-block captures.
> 2. Failed to audit train vs test covariate shift and category frequency discrepancies.
> 3. Ignored duplicate flow fingerprints with contradictory ground truth labels.

# Imports 


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

## Loading the filtered datasets

In [4]:
train_df = pd.read_csv(
    "../../data/processed/selected_training.csv"
)

test_df = pd.read_csv(
    "../../data/processed/selected_testing.csv"
)

print("Training dataset shape:", train_df.shape)
print("Testing dataset shape:", test_df.shape)

Training dataset shape: (123587, 45)
Testing dataset shape: (59543, 45)


## Dataset Overview


In [5]:
print("TRAINING DATASET")
print("=" * 50)

print("Rows:", train_df.shape[0])
print("Columns:", train_df.shape[1])

print("\nTESTING DATASET")
print("=" * 50)

print("Rows:", test_df.shape[0])
print("Columns:", test_df.shape[1])

TRAINING DATASET
Rows: 123587
Columns: 45

TESTING DATASET
Rows: 59543
Columns: 45


## Inspecting columns and data types 

In [ ]:
# compact summary of the entire dataframe 

print("\nDataset information:")
train_df.info()


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123587 entries, 0 to 123586
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 123587 non-null  int64  
 1   dur                123587 non-null  float64
 2   proto              123587 non-null  object 
 3   service            123587 non-null  object 
 4   state              123587 non-null  object 
 5   spkts              123587 non-null  int64  
 6   dpkts              123587 non-null  int64  
 7   sbytes             123587 non-null  int64  
 8   dbytes             123587 non-null  int64  
 9   rate               123587 non-null  float64
 10  sttl               123587 non-null  int64  
 11  dttl               123587 non-null  int64  
 12  sload              123587 non-null  float64
 13  dload              123587 non-null  float64
 14  sloss              123587 non-null  int64  
 15  dloss              123587 non

## Missing value analysis

In [11]:
missing_values = train_df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": (
        missing_values / len(train_df)
    ) * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

print(missing_summary)

                   Missing_Count  Missing_Percentage
id                             0                 0.0
dur                            0                 0.0
proto                          0                 0.0
service                        0                 0.0
state                          0                 0.0
spkts                          0                 0.0
dpkts                          0                 0.0
sbytes                         0                 0.0
dbytes                         0                 0.0
rate                           0                 0.0
sttl                           0                 0.0
dttl                           0                 0.0
sload                          0                 0.0
dload                          0                 0.0
sloss                          0                 0.0
dloss                          0                 0.0
sinpkt                         0                 0.0
dinpkt                         0              

## Duplicate testing 

In [ ]:
print("Duplicate records in training:", train_df.duplicated().sum())
print("Duplicate records in testing:", test_df.duplicated().sum())

Duplicate records: 0
Duplicate records in testing: 0


## Target distribution 

In [14]:
print("Training category distribution:")

print(
    train_df["attack_cat"]
    .fillna("Normal")
    .value_counts()
)

print("\nTesting category distribution:")

print(
    test_df["attack_cat"]
    .fillna("Normal")
    .value_counts()
)

Training category distribution:
attack_cat
Normal      56000
Exploits    33393
Fuzzers     18184
DoS         12264
Analysis     2000
Backdoor     1746
Name: count, dtype: int64

Testing category distribution:
attack_cat
Normal      37000
Exploits    11132
Fuzzers      6062
DoS          4089
Analysis      677
Backdoor      583
Name: count, dtype: int64


##  Numerical and categorical features

In [15]:
target_columns = ["label", "attack_cat"]

X_analysis = train_df.drop(
    columns=target_columns
)

numeric_features = X_analysis.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_analysis.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Numerical features:
['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']

Categorical features:
['proto', 'service', 'state']

Number of numerical features: 40
Number of categorical features: 3


## Constant-value features

In [18]:
constant_features = [
    column
    for column in X_analysis.columns
    if X_analysis[column].nunique(dropna=False) <= 1
]

print("Constant-value features:")
print(constant_features)

Constant-value features:
[]
